# Archive historique — selectionner_lot_pilote

Ce notebook est conservé pour documenter la préparation et les évaluations antérieures. Il ne doit pas être exécuté pour régénérer les jeux de test déjà validés.

In [ ]:
"""Fige un choix raisonné de 12 PDF et actualise leurs effectifs dans un manifeste.

Ne modifie aucune annotation ni aucun statut de validation.
Usage : python3 Scripts/pipeline_et_validation/archive_mise_au_point/selectionner_lot_pilote.ipynb
"""

import csv
from collections import Counter
from pathlib import Path

def trouver_racine() -> Path:
    courant = Path.cwd().resolve()
    for candidat in (courant, *courant.parents):
        if (candidat / "total_donnees").is_dir() and (candidat / "evaluation_extraction").is_dir():
            return candidat
    raise FileNotFoundError("Ouvre ce notebook depuis le dossier de travail.")

RACINE = trouver_racine()
DOSSIER = RACINE / "evaluation_extraction"

# Ordre de travail : calibration, cas simples, puis difficultés de structure.
SELECTION = [
    ("CDM - 2015_07_01.pdf", "calibration", "Sept nominations déjà vérifiées ; point de départ pour appliquer les consignes."),
    ("CDM - 2016_01_16.pdf", "ocr;document_negatif", "PDF initialement illisible par extraction de texte ; OCR de deux pages ; aucune nomination détectée et contrôlée."),
    ("CDM - 2017_01_05.pdf", "fonctions_individuelles", "Nominations individuelles ; vérification des noms, matricules et postes explicites."),
    ("CDM - 2024_07_10.pdf", "fonctions_individuelles;conseils_administration", "Document récent avec un petit nombre d'annonces des deux types."),
    ("CDM - 2021_06_09.pdf", "conseils_administration;roles_collectifs", "Annonces de conseils d'administration uniquement ; rôles et organismes à rattacher à chaque personne."),
    ("CDM - 2020_01_08.pdf", "fonctions_individuelles;conseils_administration;saut_de_page", "Fonctions réparties sur deux pages, notamment l'annonce d'Ousmane Yentéma YONLI."),
    ("CDM - 2023_01_11.pdf", "fonctions_individuelles;conseils_administration;renouvellements", "Nominations et renouvellements de mandats ; délimitation des rôles et institutions."),
    ("CDM - 2016_12_28.pdf", "fonctions_individuelles;conseils_administration;nominations_multiples", "Rubrique au singulier ; plusieurs décrets et deux fonctions annoncées pour Harouna KADIO."),
    ("CDM - 2018_06_18.pdf", "listes_collectives;conseils_administration;personnes_remplacees", "Listes de préfets et membres de la CNCC ; fonctions héritées d'en-têtes et personnes remplacées à exclure."),
    ("CDM -2019_10_30.pdf", "listes_collectives;ponctuation_nom", "Listes de vice-présidents ; cas Aboubakar, TRAORE à confirmer visuellement ; document plus chargé."),
    ("CDM - 2022_06_16.pdf", "revocations;date_discordante;conseils_administration", "Séance du 8 juin 2022 malgré le nom du fichier ; séparer nominations et révocations."),
    ("CDM - 2025_06_18.pdf", "fin_de_fonctions;conseils_administration", "Rubrique FIN DE FONCTIONS : vérifier l'exclusion de cette mention et la reprise des nominations après elle."),
]


def lire(nom):
    with (DOSSIER / nom).open(encoding="utf-8-sig", newline="") as fichier:
        return list(csv.DictReader(fichier))


def main():
    documents = {x["document_id"]: x for x in lire("documents_echantillon.csv")}
    suivi = {x["document_id"]: x for x in lire("suivi_annotations.csv")}
    annotations = lire("reference_annotations.csv")
    lignes = []
    for ordre, (identifiant, difficultes, motif) in enumerate(SELECTION, 1):
        document = documents[identifiant]
        for champ in ("pdf", "texte_pour_annotation"):
            if not (RACINE / document[champ]).is_file():
                raise FileNotFoundError(document[champ])
        annonces = [x for x in annotations if x["document_id"] == identifiant]
        types = Counter(x["type_nomination"] for x in annonces)
        dates = sorted({x["date_conseil"] for x in annonces})
        lignes.append({
            "ordre_relecture": ordre,
            "document_id": identifiant,
            "annee": document["annee"],
            "date_nom_fichier": document["date_nom_fichier"],
            "dates_seance_dans_grille": ";".join(dates),
            "pdf": document["pdf"],
            "texte_pour_annotation": document["texte_pour_annotation"],
            "difficultes": difficultes,
            "motif_selection": motif,
            "nombre_lignes_grille": len(annonces),
            "lignes_a_verifier": sum(x["statut_verification"] == "a_verifier" for x in annonces),
            "lignes_verifiees": sum(x["statut_verification"] == "verifie_sur_pdf" for x in annonces),
            "fonctions_individuelles": types["fonction_individuelle"],
            "conseils_administration": types["conseil_administration"],
            "pages_avec_annotations": ";".join(map(str, sorted({int(x["page"]) for x in annonces}))),
            "statut_document": suivi[identifiant]["statut"],
            "usage": "mise_au_point_et_evaluation_exploratoire",
        })
    sortie = DOSSIER / "lot_pilote_12_documents.csv"
    with sortie.open("w", encoding="utf-8-sig", newline="") as fichier:
        writer = csv.DictWriter(fichier, fieldnames=list(lignes[0]))
        writer.writeheader()
        writer.writerows(lignes)
    print(f"{len(lignes)} PDF ; {sum(x['nombre_lignes_grille'] for x in lignes)} lignes ; "
          f"{sum(x['lignes_a_verifier'] for x in lignes)} à vérifier. Manifeste : {sortie}")


## Exécution manuelle

Vérifier les chemins et les fichiers de sortie avant toute exécution.

In [ ]:
# main()